# Word LLM Translation Workflow — Fallback Translation Stage

- **Workflow stage:** `fallback_completed` (written at the end of this notebook)
- **Input checkpoint:** evaluation checkpoint from the previous stage
- **Source document:** loaded from checkpoint metadata
- **Source language:** loaded from checkpoint metadata
- **Target language:** loaded from checkpoint metadata
- **Primary translator model:** loaded from checkpoint metadata
- **Evaluator model:** loaded from checkpoint metadata
- **Fallback translator model:** user-defined in this notebook
- **Purpose of this notebook:** retranslate only the elements that failed evaluation, then reevaluate those fallback translations

## Purpose of this notebook
This notebook handles the fallback translation stage for elements that failed the evaluator in the previous notebook. It reuses the saved source language, target language, and evaluator settings from the checkpoint, applies a fallback translation model only to failed elements, and then reevaluates those fallback translations.

## Fallback translator model note

In this workflow, the fallback translation stage is configured to use an Anthropic/Claude model. Do not substitute a different provider or model API path here unless you also update the corresponding fallback-translation helper functions in `workflow_helpers.py`, since the current implementation is written for the Anthropic client and request/response pattern.

## Metadata flow
This notebook:

- loads `metadata` and `elements` from the evaluation checkpoint
- pulls inherited workflow settings from checkpoint metadata
- defines and records the fallback model name
- defines and records the fallback system message
- runs fallback translation only on evaluator-failed elements
- reevaluates fallback-translated elements using the saved evaluator settings
- writes accepted fallback results into the `final` fields
- saves an updated checkpoint with both `metadata` and `elements`

## Notes
- Fallback translation is only attempted for elements where `evaluator_passed` is `False` and `evaluator_error` is `None`.
- Reevaluation is performed after fallback translation using the same evaluator model and evaluator prompt saved in metadata.
- This notebook uses a single fallback attempt followed by a single reevaluation pass.
- Elements that still fail after fallback reevaluation remain unresolved for later review/finalization.

### Setup

In [1]:
# User-defined models for this notebook stage
claude_model = "claude-sonnet-4-6"

In [2]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints_dir = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints_dir)

Found JSON files:
- elements_batched_20260411_1519.json
- evaluation_completed_20260411_1604.json
- primary_retry_completed_1_20260411_1541.json
- primary_translation_completed_20260411_1527.json


In [3]:
# Load checkpoint state (metadata + elements)
# filename contains evaluation_completed

import os
from importlib import reload
import workflow_helpers
from pprint import pprint

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "evaluation_completed_20260411_1604.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load normalized metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Top-level state loaded through workflow_helpers.load_elements_checkpoint")
print("\nMetadata:")
pprint(metadata)

print(f"\nLoaded checkpoint with {len(elements)} elements.")
print("\nExample entry:")
pprint(elements[0] if elements else None)

Loaded checkpoint: checkpoints\evaluation_completed_20260411_1604.json
Top-level state loaded through workflow_helpers.load_elements_checkpoint

Metadata:
{'docxfilename': 'custom_word_styles_example.docx',
 'evaluation_model_name': 'gpt-5.4-mini',
 'evaluation_system_message': 'You are a translation validator for Biblical '
                              'education materials intended for a '
                              'Protestant/Evangelical audience.\n'
                              '\n'
                              'Your task is to evaluate whether each candidate '
                              'translation faithfully preserves the meaning of '
                              'the source text and preserves required Markdown '
                              '/ inline formatting.\n'
                              '\n'
                              'Be careful but not overly strict. Allow natural '
                              'translation variation. Do not fail a '
                   

In [ ]:
# Identify elements eligible for fallback translation

fallback_elements = [
    el for el in elements
    if el.get("final") is None
    and el.get("primary_translation") not in (None, "")
    and (
        el.get("evaluator_passed") is False
        or el.get("evaluator_error") is not None
    )
]

print("Total elements:", len(elements))
print("Eligible for fallback:", len(fallback_elements))

if fallback_elements:
    print("\nExample fallback element:")
    print({
        "element_id": fallback_elements[0].get("element_id"),
        "batch_number": fallback_elements[0].get("batch_number"),
        "text": fallback_elements[0].get("text"),
        "primary_translation": fallback_elements[0].get("primary_translation"),
        "evaluator_feedback": fallback_elements[0].get("evaluator_feedback"),
        "evaluator_error": fallback_elements[0].get("evaluator_error"),
    })

Total elements: 29
Eligible for fallback: 1

Example fallback element:
{'element_id': '44b22fddd655', 'batch_number': 7, 'text': 'The Bible consists of the **Hebrew** and **Greek** Scriptures.', 'primary_translation': '聖經是由**希伯來文**和**希臘文**的經卷所組成。', 'evaluator_feedback': 'The source says Hebrew and Greek Scriptures, but the translation says Hebrew and Greek languages/versions of scrolls, which changes the meaning.'}


In [5]:
# Group fallback-eligible elements by existing batch_number

fallback_batches = {}
for el in fallback_elements:
    batch_number = el.get("batch_number")
    fallback_batches.setdefault(batch_number, []).append(el)

fallback_batch_numbers = sorted(fallback_batches.keys())

print("Fallback batches detected:", len(fallback_batch_numbers))
print("Batch numbers:", fallback_batch_numbers)

print("\nBatch sizes:")
for batch_number in fallback_batch_numbers:
    print(f"  batch {batch_number}: {len(fallback_batches[batch_number])} elements")

Fallback batches detected: 1
Batch numbers: [7]

Batch sizes:
  batch 7: 1 elements


In [6]:
# Preview the fallback payload shape for one batch

preview_batch_number = fallback_batch_numbers[0]

preview_fallback_payload = {
    "elements": [
        {
            "element_id": el["element_id"],
            "source_text": el["text"],
            "previous_translation": el["primary_translation"],
            "evaluator_feedback": el["evaluator_feedback"],
        }
        for el in fallback_batches[preview_batch_number]
    ]
}

print("Preview batch number:", preview_batch_number)
print("Payload element count:", len(preview_fallback_payload["elements"]))
print("\nFirst payload item:")
print(preview_fallback_payload["elements"][0])

Preview batch number: 7
Payload element count: 1

First payload item:
{'element_id': '44b22fddd655', 'source_text': 'The Bible consists of the **Hebrew** and **Greek** Scriptures.', 'previous_translation': '聖經是由**希伯來文**和**希臘文**的經卷所組成。', 'evaluator_feedback': 'The source says Hebrew and Greek Scriptures, but the translation says Hebrew and Greek languages/versions of scrolls, which changes the meaning.'}


In [7]:
# Pull inherited workflow settings from checkpoint metadata

source_language = metadata.get("source_language")
target_language = metadata.get("target_language")
gemini_model = metadata.get("primary_model_name")
openai_model = metadata.get("evaluation_model_name")
primary_system_message = metadata.get("primary_system_message")
evaluation_system_message = metadata.get("evaluation_system_message")

missing = [
    name for name, value in {
        "source_language": source_language,
        "target_language": target_language,
        "gemini_model": gemini_model,
        "openai_model": openai_model,
        "primary_system_message": primary_system_message,
        "evaluation_system_message": evaluation_system_message,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required metadata field(s) in checkpoint: {', '.join(missing)}"
    )

# User-defined model for this notebook stage
claude_model = "claude-sonnet-4-6"

# Record fallback model in metadata
metadata["fallback_model_name"] = claude_model

print("Source language:", source_language)
print("Target language:", target_language)
print("Primary model:", gemini_model)
print("Evaluation model:", openai_model)
print("Fallback model:", claude_model)
print("Primary system message preview:", primary_system_message[:100])
print("Evaluation system message preview:", evaluation_system_message[:100])

Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview
Evaluation model: gpt-5.4-mini
Fallback model: claude-sonnet-4-6
Primary system message preview: You are a professional translator of Biblical education materials for a Protestant/Evangelical audie
Evaluation system message preview: You are a translation validator for Biblical education materials intended for a Protestant/Evangelic


## Define the fallback translator prompt
### Adapting the fallback prompt

This prompt is intentionally more detailed than the primary translation prompt because the fallback stage is corrective: it must translate the source text again while also responding to evaluator feedback about what went wrong in the earlier attempt.

Best practices when adapting it:

- keep the **source text** as the clear source of truth and treat the previous translation and evaluator feedback as correction context only
- revise the domain-specific guidance so it matches your material (for example, Biblical, legal, technical, educational, or literary content)
- preserve the JSON input/output contract unless you also update the downstream helper code that builds fallback payloads and parses fallback responses
- keep the prompt focused on **correcting real meaning or formatting problems**, not on chasing stylistic preferences
- avoid adding narrow rules aimed at a single difficult sentence or isolated edge case, since that can make the workflow less reusable
- if you use an LLM to help rewrite the prompt, review the result carefully to make sure it still prioritizes the source text over the previous translation
- after changing the prompt, run a one-batch fallback smoke test and reevaluation smoke test before launching the full fallback pass

In general, it is fine to revise the domain guidance and correction priorities, but be cautious about changing the response schema, field names, or strict JSON-only output rules unless you are also updating the downstream fallback helper logic in `workflow_helpers.py`.

In [8]:
# Fallback translation system prompt

fallback_system_message = f"""
You are a professional translator of Biblical education materials intended for a Protestant/Evangelical audience.

Your task is to produce a corrected translation in {target_language} for each input element.

You will receive one JSON object with this structure:

{{
  "elements": [
    {{
      "element_id": "<string>",
      "source_text": "<source text in {source_language}>",
      "previous_translation": "<earlier translation attempt in {target_language}>",
      "evaluator_feedback": "<brief explanation of why the earlier translation failed evaluation>"
    }}
  ]
}}

Your job is to translate the original source text faithfully while explicitly correcting the problem identified in the evaluator feedback.

SOURCE OF TRUTH
- The source text is authoritative.
- The previous translation is provided only as context for what was attempted before.
- The evaluator feedback identifies what went wrong and what must be corrected.
- If the previous translation conflicts with the source text or evaluator feedback, follow the source text and the evaluator feedback.

REQUIREMENTS

1) Meaning preservation
   - Translate the full meaning of the source text accurately.
   - Do not omit, add, weaken, or distort meaning.
   - Correct the specific semantic problem identified in the evaluator feedback.
   - Do not repeat the same error in the new translation.

2) Formatting preservation
   - Preserve required Markdown and inline formatting from the source text.
   - This includes headings, list markers, blockquotes, backticks, bold, italics, and link syntax where applicable.
   - Preserve balanced formatting markers and structure.

3) Non-translatable content
   - Keep inline code, URLs, and other clearly literal / non-translatable content unchanged unless the source itself changes them.

4) Biblical and theological wording
   - Translate Biblical and theological terms according to their meaning in context.
   - Do not replace references to Scriptures with references to languages, books, or related concepts unless the source text explicitly means that.
   - Use standard {target_language} Protestant usage where appropriate.

5) Correction focus
   - Use the evaluator feedback to make a better corrected translation, not to write commentary about the correction.
   - Produce the best corrected translation directly.
   - Do not explain your reasoning.

6) Output discipline
   - Return only valid JSON.
   - Do not include explanations, notes, or commentary outside the JSON.
   - Do not wrap the JSON in code fences.

Return exactly one JSON object with this structure:

{{
  "elements": [
    {{
      "element_id": "<copy from input>",
      "translated_text": "<corrected translation in {target_language}>"
    }}
  ]
}}

Output rules:
- Preserve input order.
- Include exactly one output object for each input element.
- Use the same `element_id` values as the input.
- Do not include any extra top-level keys.
""".strip()

metadata["fallback_system_message"] = fallback_system_message
metadata["fallback_model_name"] = claude_model

print("Fallback system prompt defined and stored in metadata.")
print("Prompt preview:")
print(fallback_system_message[:500])

Fallback system prompt defined and stored in metadata.
Prompt preview:
You are a professional translator of Biblical education materials intended for a Protestant/Evangelical audience.

Your task is to produce a corrected translation in Traditional Chinese for each input element.

You will receive one JSON object with this structure:

{
  "elements": [
    {
      "element_id": "<string>",
      "source_text": "<source text in English>",
      "previous_translation": "<earlier translation attempt in Traditional Chinese>",
      "evaluator_feedback": "<brief explana


In [9]:
# record fallback system message and model name to metadata header
metadata["fallback_system_message"] = fallback_system_message
metadata["fallback_model_name"] = claude_model

In [10]:
print("Fallback model:", metadata.get("fallback_model_name"))
print("Fallback system message preview:", metadata.get("fallback_system_message", "")[:100])

Fallback model: claude-sonnet-4-6
Fallback system message preview: You are a professional translator of Biblical education materials intended for a Protestant/Evangeli


In [11]:
# Smoke test: run fallback translation on one failed batch without mutating the full `elements` list

from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

test_fallback_elements = [el.copy() for el in elements if el.get("batch_number") == 7]

test_fallback_elements = workflow_helpers.run_fallback_translation_pass(
    elements=test_fallback_elements,
    claude_model_name=claude_model,
    fallback_system_message=fallback_system_message,
    verbose=True,
    print_status_every_n_batches=1,
)

print("Example fallback result:")
for el in test_fallback_elements:
    if el.get("evaluator_passed") is False:
        print({
            "element_id": el.get("element_id"),
            "text": el.get("text"),
            "primary_translation": el.get("primary_translation"),
            "evaluator_feedback": el.get("evaluator_feedback"),
            "fallback_translation": el.get("fallback_translation"),
            "fallback_translation_model": el.get("fallback_translation_model"),
            "fallback_error": el.get("fallback_error"),
        })
        break

[16:17:01] Starting fallback translation: 1 batches detected.
[16:17:03] Progress: 1/1 batches completed.
[16:17:03] Fallback translation complete: 1/1 batches processed in 00:00.
Example fallback result:
{'element_id': '44b22fddd655', 'text': 'The Bible consists of the **Hebrew** and **Greek** Scriptures.', 'primary_translation': '聖經是由**希伯來文**和**希臘文**的經卷所組成。', 'evaluator_feedback': 'The source says Hebrew and Greek Scriptures, but the translation says Hebrew and Greek languages/versions of scrolls, which changes the meaning.', 'fallback_translation': '聖經由**希伯來**經卷和**希臘**經卷所組成。', 'fallback_translation_model': 'claude-sonnet-4-6', 'fallback_error': None}


In [12]:
# Quick QA on the fallback smoke test

fallback_done = sum(el.get("fallback_translation") is not None for el in test_fallback_elements)
fallback_errors = sum(el.get("fallback_error") is not None for el in test_fallback_elements)

print("Fallback translations produced:", fallback_done)
print("Fallback errors:", fallback_errors)

print("\nDetailed fallback state:")
for el in test_fallback_elements:
    if el.get("evaluator_passed") is False:
        print({
            "element_id": el.get("element_id"),
            "fallback_translation": el.get("fallback_translation"),
            "fallback_error": el.get("fallback_error"),
        })

Fallback translations produced: 1
Fallback errors: 0

Detailed fallback state:
{'element_id': '44b22fddd655', 'fallback_translation': '聖經由**希伯來**經卷和**希臘**經卷所組成。', 'fallback_error': None}


In [13]:
# Prepare fallback-translated items for reevaluation using the existing evaluator path

test_fallback_for_eval = workflow_helpers.prepare_fallback_elements_for_reevaluation(
    test_fallback_elements
)

print("Prepared reevaluation example:")
for el in test_fallback_for_eval:
    if el.get("fallback_translation") is not None:
        print({
            "element_id": el.get("element_id"),
            "primary_translation_used_for_eval": el.get("primary_translation"),
            "fallback_translation": el.get("fallback_translation"),
            "evaluator_ran": el.get("evaluator_ran"),
            "evaluator_passed": el.get("evaluator_passed"),
            "evaluator_feedback": el.get("evaluator_feedback"),
            "evaluator_error": el.get("evaluator_error"),
        })
        break

Prepared reevaluation example:
{'element_id': '44b22fddd655', 'primary_translation_used_for_eval': '聖經由**希伯來**經卷和**希臘**經卷所組成。', 'fallback_translation': '聖經由**希伯來**經卷和**希臘**經卷所組成。', 'evaluator_ran': None, 'evaluator_passed': None, 'evaluator_feedback': None, 'evaluator_error': None}


In [14]:
# Smoke test: reevaluate the fallback-translated batch

test_fallback_for_eval = workflow_helpers.run_evaluation_pass(
    elements=test_fallback_for_eval,
    openai_model_name=openai_model,
    evaluator_system_message=evaluation_system_message,
    verbose=True,
    print_status_every_n_batches=1,
)

print("Reevaluation result:")
for el in test_fallback_for_eval:
    if el.get("fallback_translation") is not None:
        print({
            "element_id": el.get("element_id"),
            "fallback_translation": el.get("fallback_translation"),
            "evaluator_passed": el.get("evaluator_passed"),
            "evaluator_feedback": el.get("evaluator_feedback"),
            "evaluator_error": el.get("evaluator_error"),
        })
        break

[16:17:03] Starting evaluation: 1 batches detected.
[16:17:06] Progress: 1/1 batches completed.
[16:17:06] Evaluation complete: 1/1 batches processed in 00:00.
Reevaluation result:
{'element_id': '44b22fddd655', 'fallback_translation': '聖經由**希伯來**經卷和**希臘**經卷所組成。', 'evaluator_passed': True, 'evaluator_feedback': '', 'evaluator_error': None}


### Run full set

In [15]:
# Run fallback translation across all failed evaluator items

from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

elements = workflow_helpers.run_fallback_translation_pass(
    elements=elements,
    claude_model_name=claude_model,
    fallback_system_message=fallback_system_message,
    verbose=True,
    print_status_every_n_batches=1,
)

[16:17:06] Starting fallback translation: 12 batches detected.
[16:17:08] Progress: 7/12 batches completed.
[16:17:08] Fallback translation complete: 12/12 batches processed in 00:00.


In [16]:
# Quick QA after fallback translation pass

fallback_done = sum(el.get("fallback_translation") is not None for el in elements)
fallback_errors = sum(el.get("fallback_error") is not None for el in elements)

print("Fallback translations produced:", fallback_done)
print("Fallback errors:", fallback_errors)

print("\nFallback-translated items:")
for el in elements:
    if el.get("fallback_translation") is not None:
        print({
            "element_id": el.get("element_id"),
            "text": el.get("text"),
            "primary_translation": el.get("primary_translation"),
            "fallback_translation": el.get("fallback_translation"),
            "fallback_error": el.get("fallback_error"),
        })

Fallback translations produced: 1
Fallback errors: 0

Fallback-translated items:
{'element_id': '44b22fddd655', 'text': 'The Bible consists of the **Hebrew** and **Greek** Scriptures.', 'primary_translation': '聖經是由**希伯來文**和**希臘文**的經卷所組成。', 'fallback_translation': '聖經由**希伯來**經典和**希臘**經典所組成。', 'fallback_error': None}


In [17]:
# Prepare fallback-translated items for reevaluation

fallback_for_eval = workflow_helpers.prepare_fallback_elements_for_reevaluation(
    elements
)

print("Prepared reevaluation copy.")
print("Example fallback-prepared item:")
for el in fallback_for_eval:
    if el.get("fallback_translation") is not None:
        print({
            "element_id": el.get("element_id"),
            "primary_translation_used_for_eval": el.get("primary_translation"),
            "fallback_translation": el.get("fallback_translation"),
            "evaluator_ran": el.get("evaluator_ran"),
            "evaluator_passed": el.get("evaluator_passed"),
            "evaluator_feedback": el.get("evaluator_feedback"),
            "evaluator_error": el.get("evaluator_error"),
        })
        break

Prepared reevaluation copy.
Example fallback-prepared item:
{'element_id': '44b22fddd655', 'primary_translation_used_for_eval': '聖經由**希伯來**經典和**希臘**經典所組成。', 'fallback_translation': '聖經由**希伯來**經典和**希臘**經典所組成。', 'evaluator_ran': None, 'evaluator_passed': None, 'evaluator_feedback': None, 'evaluator_error': None}


In [18]:
# Reevaluate all fallback-translated items

fallback_for_eval = workflow_helpers.run_evaluation_pass(
    elements=fallback_for_eval,
    openai_model_name=openai_model,
    evaluator_system_message=evaluation_system_message,
    verbose=True,
    print_status_every_n_batches=1,
)

[16:17:08] Starting evaluation: 12 batches detected.
[16:17:09] Progress: 7/12 batches completed.
[16:17:09] Evaluation complete: 12/12 batches processed in 00:00.


In [19]:
# Quick QA after fallback reevaluation

reevaluated_count = sum(
    el.get("fallback_translation") is not None and el.get("evaluator_ran") is True
    for el in fallback_for_eval
)
reeval_passed_count = sum(
    el.get("fallback_translation") is not None and el.get("evaluator_passed") is True
    for el in fallback_for_eval
)
reeval_failed_count = sum(
    el.get("fallback_translation") is not None and el.get("evaluator_passed") is False
    for el in fallback_for_eval
)

print("Fallback items reevaluated:", reevaluated_count)
print("Fallback items passed reevaluation:", reeval_passed_count)
print("Fallback items failed reevaluation:", reeval_failed_count)

print("\nDetailed fallback reevaluation results:")
for el in fallback_for_eval:
    if el.get("fallback_translation") is not None:
        print({
            "element_id": el.get("element_id"),
            "fallback_translation": el.get("fallback_translation"),
            "evaluator_passed": el.get("evaluator_passed"),
            "evaluator_feedback": el.get("evaluator_feedback"),
            "evaluator_error": el.get("evaluator_error"),
        })

Fallback items reevaluated: 1
Fallback items passed reevaluation: 1
Fallback items failed reevaluation: 0

Detailed fallback reevaluation results:
{'element_id': '44b22fddd655', 'fallback_translation': '聖經由**希伯來**經典和**希臘**經典所組成。', 'evaluator_passed': True, 'evaluator_feedback': '', 'evaluator_error': None}


In [20]:
# Merge fallback reevaluation results back into canonical elements
# and assign final/final_model only for items that passed after fallback

fallback_eval_by_id = {
    el["element_id"]: el
    for el in fallback_for_eval
    if el.get("fallback_translation") is not None
}

for el in elements:
    matched = fallback_eval_by_id.get(el["element_id"])
    if not matched:
        continue

    # carry reevaluation results back
    el["evaluator_ran"] = matched.get("evaluator_ran")
    el["evaluator_passed"] = matched.get("evaluator_passed")
    el["evaluator_feedback"] = matched.get("evaluator_feedback")
    el["evaluator_error"] = matched.get("evaluator_error")

    # assign final only if fallback passed reevaluation
    if matched.get("evaluator_passed") is True:
        el["final"] = el.get("fallback_translation")
        el["final_model"] = el.get("fallback_translation_model")

In [21]:
# Inspect final state of fallback-targeted items

for el in elements:
    if el.get("fallback_translation") is not None:
        print({
            "element_id": el.get("element_id"),
            "primary_translation": el.get("primary_translation"),
            "fallback_translation": el.get("fallback_translation"),
            "evaluator_passed": el.get("evaluator_passed"),
            "evaluator_feedback": el.get("evaluator_feedback"),
            "final": el.get("final"),
            "final_model": el.get("final_model"),
        })

{'element_id': '44b22fddd655', 'primary_translation': '聖經是由**希伯來文**和**希臘文**的經卷所組成。', 'fallback_translation': '聖經由**希伯來**經典和**希臘**經典所組成。', 'evaluator_passed': True, 'evaluator_feedback': '', 'final': '聖經由**希伯來**經典和**希臘**經典所組成。', 'final_model': 'claude-sonnet-4-6'}


In [22]:
metadata["stage"] = workflow_helpers.WORKFLOW_STAGES["fallback_completed"]

In [23]:
# Save fallback-completed checkpoint

from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

checkpoint_path = workflow_helpers.save_elements_checkpoint(
    elements=elements,
    base_filename=workflow_helpers.WORKFLOW_STAGES["fallback_completed"],
    metadata=metadata,
)

print("Checkpoint saved to:", checkpoint_path)
print("Metadata saved:")
print(metadata)

Checkpoint saved to: checkpoints\fallback_completed_20260411_1617.json
Metadata saved:
{'stage': 'fallback_completed', 'docxfilename': 'custom_word_styles_example.docx', 'source_language': 'English', 'target_language': 'Traditional Chinese', 'primary_model_name': 'gemini-3.1-pro-preview', 'evaluation_model_name': 'gpt-5.4-mini', 'fallback_model_name': 'claude-sonnet-4-6', 'primary_system_message': 'You are a professional translator of Biblical education materials for a Protestant/Evangelical audience.\n\nYou will receive a single JSON object with the following structure:\n\n{\n  "elements": [\n    {\n      "id": "<string>",\n      "text": "<chunked Markdown in English>"\n    },\n    ...\n  ]\n}\n\nEach `text` field is a small chunk of Markdown in English. For each element, you must translate the natural-language prose into Traditional Chinese while preserving formatting and structure.\n\nYou must respond with a single valid JSON object of the form:\n\n{\n  "elements": [\n    {\n      "

## Notebook checkpoint and handoff

This notebook completed the fallback translation stage of the Word-to-LLM workflow by performing the following steps:

- loaded the prior checkpoint containing metadata, primary translations, and evaluator results
- pulled `source_language`, `target_language`, `primary_model_name`, and `evaluation_model_name` forward from checkpoint metadata
- loaded the saved evaluator system message from checkpoint metadata for fallback reevaluation
- defined the fallback model for this stage
- defined the fallback system prompt directly in the notebook
- saved the fallback system prompt into checkpoint metadata for provenance
- identified all elements that failed evaluation and were eligible for fallback translation
- reused the existing `batch_number` values for batched fallback requests
- ran a one-batch fallback smoke test and reevaluation smoke test
- executed the full fallback translation pass across all eligible failed items
- reevaluated the fallback-translated items using the evaluator model
- assigned `final` and `final_model` for fallback items that passed reevaluation
- updated workflow metadata to reflect the completed fallback stage
- saved the resulting intermediate state to a timestamped JSON checkpoint together with workflow metadata

### Output of this notebook
The main output is a timestamped JSON checkpoint containing:

- a top-level `metadata` block
- an `elements` list containing primary translation results, evaluator results, fallback translations, and final accepted results where applicable

This file is intended to be used as input for the next notebook.

### Metadata saved at this stage
The checkpoint metadata currently records:

- `stage = fallback_completed`
- `docxfilename`
- `source_language`
- `target_language`
- `primary_model_name`
- `evaluation_model_name`
- `fallback_model_name`
- `primary_system_message`
- `evaluation_system_message`
- `fallback_system_message`

### Scope of this notebook
This notebook focuses only on fallback translation and reevaluation of previously failed items.

At this stage:

- `fallback_translation` stores the fallback model output
- `fallback_translation_model` stores the fallback model name
- `fallback_error` stores any fallback-stage errors
- fallback outputs are reevaluated before acceptance
- `final` and `final_model` are assigned only for fallback items that pass reevaluation
- items that still fail after fallback reevaluation remain unresolved for later review/finalization

### Next step
The next notebook can load the fallback-completed checkpoint, inspect the final accepted results, identify any unresolved items, and produce final output artifacts.

Go to: `6_finalize_translation.ipynb`